# Random ELFNet structures with void markers

Select five random structures from each dataset category (`2` and `Mono`), requiring a different detected space group for every selected structure. Run the geometric void finder and write one VESTA-readable CIF per structure under `elfnet_data_analysis/vesta_void_cifs/`. Void centers are helium marker sites labeled `Void_1`, `Void_2`, etc.

In [1]:
import importlib
import random
import re
import sys
from pathlib import Path

import numpy as np
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# Find the repository root from the notebook kernel's current directory.
search_dirs = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    path for path in search_dirs
    if (path / 'void_detection' / 'find_voronoi_voids.py').is_file()
)
dataset_root = (
    repo_root
    / 'elfnet_data'
    / 'extracted'
    / 'dft-reference'
    / 'dft-reference-elfs-50000-v1'
)
output_dir = repo_root / 'elfnet_data_analysis' / 'vesta_void_cifs'
output_dir.mkdir(parents=True, exist_ok=True)

finder_dir = repo_root / 'void_detection'
if str(finder_dir) not in sys.path:
    sys.path.insert(0, str(finder_dir))
import find_voronoi_voids
importlib.reload(find_voronoi_voids)
find_candidate_voids = find_voronoi_voids.find_candidate_voids


In [2]:
# The fixed seed makes this random sample reproducible. Space-group numbers must
# be unique across both categories, not just within each category.
random_seed = 20261019
sample_per_category = 5
rng = random.Random(random_seed)
used_spacegroups = set()
selected_systems = []

for category in ('2', 'Mono'):
    category_dir = dataset_root / category
    if not category_dir.is_dir():
        raise FileNotFoundError(f'Dataset category not found: {category_dir}')

    poscar_paths = list(category_dir.rglob('POSCAR'))
    rng.shuffle(poscar_paths)
    selected_in_category = 0

    for poscar_path in poscar_paths:
        structure = Structure.from_file(str(poscar_path))
        analyzer = SpacegroupAnalyzer(structure, symprec=0.01)
        spacegroup_symbol = analyzer.get_space_group_symbol()
        spacegroup_number = analyzer.get_space_group_number()
        if spacegroup_number in used_spacegroups:
            continue

        used_spacegroups.add(spacegroup_number)
        selected_systems.append({
            'category': category,
            'poscar_path': poscar_path,
            'structure_id': poscar_path.parent.relative_to(dataset_root).as_posix(),
            'structure': structure,
            'spacegroup_symbol': spacegroup_symbol,
            'spacegroup_number': int(spacegroup_number),
        })
        selected_in_category += 1
        print(
            f"Selected {category}: {selected_systems[-1]['structure_id']} "
            f"(space group {spacegroup_symbol}, #{spacegroup_number})"
        )
        if selected_in_category == sample_per_category:
            break

    if selected_in_category != sample_per_category:
        raise RuntimeError(
            f'Found only {selected_in_category} structures with distinct space groups '
            f'in category {category}; {sample_per_category} requested.'
        )

assert len(selected_systems) == 10
assert len({item['spacegroup_number'] for item in selected_systems}) == 10


Selected 2: 2/Ti2H13_Immm_200/Mn_Pr (space group Pmmm, #47)
Selected 2: 2/ZrH8_I4mmm_200/Sc_Pr (space group P4/mmm, #123)
Selected 2: 2/YH12_C2m_200/Cu_Pd (space group P2/m, #10)
Selected 2: 2/YH9_P63m_100GPa_200/Tc_Tb (space group P-6m2, #187)
Selected 2: 2/AcH5_C2m_50GPa_200/Cu_Eu (space group Cm, #8)
Selected Mono: Mono/CaH6_Im-3m_200/Sr (space group Im-3m, #229)
Selected Mono: Mono/BaH8_Imma_200/Ga (space group P6/mmm, #191)
Selected Mono: Mono/SrH10_P21c_200/W (space group Immm, #71)
Selected Mono: Mono/ThH7_P21c_200/In (space group P-1, #2)
Selected Mono: Mono/LaH11_P4nmm_200/Ce (space group P4/nmm, #129)


In [3]:
def write_vesta_cif(structure, voids, output_path, data_name):
    """Write the structure and labeled helium void markers as a CIF."""
    lattice = structure.lattice
    cif_lines = [
        f'data_{data_name}',
        '# Candidate void positions are helium marker sites.',
        f'_cell_length_a {lattice.a:.8f}',
        f'_cell_length_b {lattice.b:.8f}',
        f'_cell_length_c {lattice.c:.8f}',
        f'_cell_angle_alpha {lattice.alpha:.8f}',
        f'_cell_angle_beta {lattice.beta:.8f}',
        f'_cell_angle_gamma {lattice.gamma:.8f}',
        "_space_group_name_H-M_alt 'P 1'",
        '_space_group_IT_number 1',
        'loop_',
        '_atom_site_label',
        '_atom_site_type_symbol',
        '_atom_site_fract_x',
        '_atom_site_fract_y',
        '_atom_site_fract_z',
        '_atom_site_occupancy',
    ]

    for index, site in enumerate(structure, start=1):
        x, y, z = site.frac_coords % 1.0
        element = site.specie.symbol
        cif_lines.append(f'{element}{index} {element} {x:.8f} {y:.8f} {z:.8f} 1.0')

    for index, void in enumerate(voids, start=1):
        x, y, z = np.asarray(void['fractional_coords'], dtype=float) % 1.0
        cif_lines.append(f'Void_{index} He {x:.8f} {y:.8f} {z:.8f} 1.0')

    output_path.write_text('\n'.join(cif_lines) + '\n')


created_cifs = []
for index, item in enumerate(selected_systems, start=1):
    print(f"Finding voids for {index}/10: {item['structure_id']}")
    voids = find_candidate_voids(item['structure'])
    path_slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', item['structure_id'])
    cif_name = (
        f"{item['category']}__{path_slug}__SG{item['spacegroup_number']}.cif"
    )
    cif_path = output_dir / cif_name
    write_vesta_cif(item['structure'], voids, cif_path, cif_path.stem)
    created_cifs.append({
        'category': item['category'],
        'structure_id': item['structure_id'],
        'spacegroup': item['spacegroup_symbol'],
        'spacegroup_number': item['spacegroup_number'],
        'atom_count': len(item['structure']),
        'void_marker_count': len(voids),
        'cif_path': cif_path,
    })
    print(f"  {len(voids)} void markers -> {cif_path}")

print(f'Created {len(created_cifs)} CIF files in {output_dir}')


Finding voids for 1/10: 2/Ti2H13_Immm_200/Mn_Pr


  4 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/2__2_Ti2H13_Immm_200_Mn_Pr__SG47.cif
Finding voids for 2/10: 2/ZrH8_I4mmm_200/Sc_Pr
  2 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/2__2_ZrH8_I4mmm_200_Sc_Pr__SG123.cif
Finding voids for 3/10: 2/YH12_C2m_200/Cu_Pd
  3 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/2__2_YH12_C2m_200_Cu_Pd__SG10.cif
Finding voids for 4/10: 2/YH9_P63m_100GPa_200/Tc_Tb


  1 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/2__2_YH9_P63m_100GPa_200_Tc_Tb__SG187.cif
Finding voids for 5/10: 2/AcH5_C2m_50GPa_200/Cu_Eu


  7 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/2__2_AcH5_C2m_50GPa_200_Cu_Eu__SG8.cif
Finding voids for 6/10: Mono/CaH6_Im-3m_200/Sr
  1 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/Mono__Mono_CaH6_Im-3m_200_Sr__SG229.cif
Finding voids for 7/10: Mono/BaH8_Imma_200/Ga
  1 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/Mono__Mono_BaH8_Imma_200_Ga__SG191.cif
Finding voids for 8/10: Mono/SrH10_P21c_200/W


  2 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/Mono__Mono_SrH10_P21c_200_W__SG71.cif
Finding voids for 9/10: Mono/ThH7_P21c_200/In
  4 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/Mono__Mono_ThH7_P21c_200_In__SG2.cif
Finding voids for 10/10: Mono/LaH11_P4nmm_200/Ce
  3 void markers -> /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs/Mono__Mono_LaH11_P4nmm_200_Ce__SG129.cif
Created 10 CIF files in /Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor/elfnet_data_analysis/vesta_void_cifs
